# Boston Housing — Análisis Exploratorio de Datos 

## ¿Qué queremos descubrir antes de construir un modelo?

El objetivo de este notebook es **entender el conjunto de datos antes de intentar predecir `MEDV`**. En lugar de comenzar con matrices extensas y cálculos manuales, utilizaremos **Pandas** para organizar y resumir los datos y **Seaborn** para convertir los resultados en representaciones visuales fáciles de interpretar.

### Ruta del análisis

1. **Conocer los datos:** qué representa cada fila y cada variable.
2. **Verificar la calidad:** faltantes, duplicados y número de valores distintos.
3. **Reconocer el tipo de las variables:** continuas, discretas y binarias.
4. **Explorar distribuciones:** centro, dispersión, asimetría y valores extremos.
5. **Estudiar `MEDV`:** comportamiento de la variable que posteriormente queremos predecir.
6. **Relacionar los predictores con `MEDV`:** dirección, intensidad y forma de las asociaciones.
7. **Relacionar los predictores entre sí:** identificar información potencialmente redundante.
8. **Sintetizar hallazgos:** decidir qué debe tenerse presente antes del modelado.

> **Idea central:** un EDA no consiste en producir muchas tablas. Consiste en formular preguntas sobre los datos y utilizar estadísticas y visualizaciones para responderlas.

## 1. Conjunto de datos y significado de las variables

Cada fila representa una zona del área de Boston. La variable objetivo será **`MEDV`**, el valor mediano de las viviendas ocupadas por sus propietarios, expresado históricamente en miles de dólares.

### Variables

- **`CRIM`**: tasa de criminalidad per cápita.
- **`ZN`**: proporción de suelo residencial destinado a lotes grandes.
- **`INDUS`**: proporción de superficie dedicada a negocios no minoristas.
- **`CHAS`**: indicador binario de cercanía al río Charles (`0`/`1`).
- **`NOX`**: concentración de óxidos de nitrógeno.
- **`RM`**: número promedio de habitaciones por vivienda.
- **`AGE`**: proporción de viviendas construidas antes de 1940.
- **`DIS`**: distancia ponderada a centros de empleo de Boston.
- **`RAD`**: índice discreto de accesibilidad a autopistas radiales.
- **`TAX`**: tasa de impuesto a la propiedad.
- **`PTRATIO`**: relación alumno–docente.
- **`B`**: transformación histórica basada en composición racial de la población.
- **`LSTAT`**: porcentaje de población clasificada históricamente como de menor estatus socioeconómico.
- **`MEDV`**: valor mediano de las viviendas; será nuestra variable objetivo.

> **Nota de contexto.** Boston Housing es un conjunto histórico muy utilizado en docencia, pero contiene variables y supuestos que hoy requieren una lectura ética cuidadosa. En este notebook `B` se conserva únicamente para reproducir el conjunto original; no se interpretará como una variable causal ni normativa.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## 2. Preparación del entorno y carga de datos

### Pregunta
**¿Podemos cargar la base de manera simple y trabajar con nombres de variables legibles?**

La ventaja de Pandas es que cada columna conserva su nombre. Así evitamos trabajar con índices numéricos como `datos[:, 5]` y podemos escribir directamente `df['RM']`.

In [ ]:
# El archivo contiene la misma estructura utilizada en el notebook original.
df = pd.read_csv("./boston.csv")
df.columns = df.columns.str.upper()

print(f"Observaciones: {df.shape[0]}")
print(f"Variables:     {df.shape[1]}")
df.head(4)

### Primera lectura

El conjunto estándar contiene **506 observaciones y 14 variables**. Una tabla pequeña al inicio es útil para verificar que la carga fue correcta; a partir de aquí priorizaremos las visualizaciones.

## 3. Calidad e integridad de los datos

### Pregunta
**¿Hay problemas evidentes que debamos resolver antes de interpretar las variables?**

Revisaremos tres aspectos básicos: valores faltantes, filas duplicadas y cantidad de valores diferentes por variable.

In [ ]:
print("Valores faltantes totales:", int(df.isna().sum().sum()))
print("Filas duplicadas:         ", int(df.duplicated().sum()))

En la versión estándar no aparecen valores faltantes ni filas duplicadas. Eso no significa que el conjunto sea perfecto: todavía debemos revisar **tipos de variables, distribuciones, valores extremos y relaciones**.

### ¿Todas las columnas numéricas son realmente continuas?

El número de valores distintos es una forma rápida de detectar variables binarias o discretas.

In [ ]:
unicos = (
    df.nunique()
      .sort_values()
      .rename("Valores distintos")
      .reset_index().rename(columns={"index": "Variable"})
)

plt.figure(figsize=(9, 5))
sns.barplot(data=unicos, x="Valores distintos", y="Variable")
plt.title("Número de valores distintos por variable")
plt.xlabel("Cantidad de valores distintos")
plt.ylabel("")
plt.show()

### Interpretación

- `CHAS` tiene solo **dos valores**: debe interpretarse como una variable **binaria**.
- `RAD` tiene pocos niveles: funciona mejor como **índice discreto** que como una magnitud continua.
- La mayoría de las demás variables presentan muchos valores diferentes y pueden explorarse inicialmente como cuantitativas.

Esto es importante porque un histograma o una correlación no deben interpretarse de la misma manera para todos los tipos de variable.

## 4. Variables discretas y binarias

### Pregunta
**¿Cómo están distribuidas las observaciones entre las categorías de `CHAS` y los niveles de `RAD`?**

Para estas variables es más informativo contar observaciones que estimar una densidad continua.

In [ ]:
discretas = df[["CHAS", "RAD"]].melt(
    var_name="Variable", value_name="Valor"
)

g = sns.catplot(
    data=discretas,
    x="Valor",
    col="Variable",
    kind="count",
    sharex=False,
    sharey=False,
    height=3.5,
    aspect=1.25
)
g.set_axis_labels("Valor", "Número de observaciones")
g.set_titles("{col_name}")
g.fig.suptitle("Distribución de las variables discretas", y=1.05)
plt.show()

### Interpretación

`CHAS` está claramente desbalanceada hacia el valor `0`. En `RAD` también existen niveles mucho más frecuentes que otros. Estas diferencias deben recordarse cuando más adelante comparemos `MEDV` entre grupos.

## 5. Distribución de las variables cuantitativas

### Pregunta
**¿Dónde se concentran los datos y qué tan simétricas o dispersas son las variables?**

Un histograma permite ver la forma de la distribución. La curva suavizada ayuda a reconocer concentraciones y colas, pero no reemplaza al histograma.

In [ ]:
cuantitativas = [
    "CRIM", "ZN", "INDUS", "NOX", "RM", "AGE",
    "DIS", "TAX", "PTRATIO", "B", "LSTAT", "MEDV"
]

long = df[cuantitativas].melt(
    var_name="Variable", value_name="Valor"
)

g = sns.displot(
    data=long,
    x="Valor",
    col="Variable",
    col_wrap=3,
    bins=25,
    kde=True,
    facet_kws={"sharex": False, "sharey": False},
    height=2.6,
    aspect=1.25
)
g.set_axis_labels("Valor", "Frecuencia")
g.set_titles("{col_name}")
g.fig.suptitle("Distribuciones de las variables cuantitativas", y=1.02)
plt.show()

### ¿Qué patrones aparecen?

- **`CRIM`** tiene una cola muy larga hacia valores altos: su distribución es fuertemente asimétrica.
- **`ZN`** concentra una gran cantidad de observaciones en cero.
- **`RM`** es considerablemente más compacta y cercana a una forma unimodal.
- **`AGE`** presenta una concentración importante en valores altos.
- **`MEDV`** no es completamente simétrica y muestra una acumulación visible en el extremo superior.

La diferencia entre estas formas ya anticipa que un mismo tratamiento estadístico no será apropiado para todas las variables.

## 6. Dispersión y posibles valores extremos

### Pregunta
**¿Qué observaciones se alejan del comportamiento central de cada variable?**

El boxplot resume visualmente la mediana, los cuartiles y la regla exploratoria de `1.5 × IQR`. Los puntos fuera de los bigotes son **candidatos a valores atípicos**, no errores automáticos.

In [ ]:
g = sns.catplot(
    data=long,
    x="Valor",
    col="Variable",
    col_wrap=3,
    kind="box",
    sharex=False,
    sharey=False,
    height=2.35,
    aspect=1.3
)
g.set_axis_labels("Valor", "")
g.set_titles("{col_name}")
g.fig.suptitle("Boxplots: dispersión y observaciones extremas", y=1.02)
plt.show()

### Interpretación

Variables como `CRIM`, `ZN`, `B` y `LSTAT` muestran numerosos puntos alejados del rango intercuartílico. Antes de eliminarlos habría que responder una pregunta fundamental:

> **¿son errores de medición o zonas reales con características poco frecuentes?**

El EDA solo los señala; la decisión de conservarlos, transformarlos o excluirlos necesita contexto.

### 6.1 Caso de estudio: `CRIM`

`CRIM` es un buen ejemplo para conectar el boxplot con la regla del rango intercuartílico.

In [ ]:
q1 = df["CRIM"].quantile(0.25)
q3 = df["CRIM"].quantile(0.75)
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr
n_extremos = (df["CRIM"] > limite_superior).sum()

print(f"Q1:              {q1:.3f}")
print(f"Mediana:         {df['CRIM'].median():.3f}")
print(f"Q3:              {q3:.3f}")
print(f"Límite superior: {limite_superior:.3f}")
print(f"Puntos superiores al límite: {n_extremos}")

plt.figure(figsize=(9, 3.5))
sns.boxplot(data=df, x="CRIM")
plt.title("CRIM: el rango central ocupa una pequeña parte de la escala total")
plt.xlabel("CRIM")
plt.show()

El boxplot deja claro por qué la media de `CRIM` puede ser poco representativa del comportamiento típico: unos pocos valores altos estiran fuertemente la escala.

## 7. Análisis de la variable objetivo: `MEDV`

### Pregunta
**¿Cómo se distribuye aquello que posteriormente queremos predecir?**

Antes de estudiar predictores conviene entender la variable objetivo por sí sola.

In [ ]:
medv_max = df["MEDV"].max()
n_max = df["MEDV"].eq(medv_max).sum()

plt.figure(figsize=(9, 4.5))
sns.histplot(data=df, x="MEDV", bins=25, kde=True)
plt.axvline(medv_max, linestyle="--", linewidth=2)
plt.title(f"Distribución de MEDV — {n_max} observaciones exactamente en {medv_max:g}")
plt.xlabel("MEDV")
plt.ylabel("Frecuencia")
plt.show()

print(f"Mediana de MEDV: {df['MEDV'].median():.2f}")
print(f"Media de MEDV:   {df['MEDV'].mean():.2f}")
print(f"Máximo observado:{medv_max:6.2f}")

### Hallazgo importante

En el conjunto estándar aparecen **16 observaciones exactamente en `MEDV = 50`**. La acumulación en un mismo valor máximo sugiere un **efecto de techo** en el registro histórico. Un modelo posterior no debería interpretar ese límite como si fuera simplemente otro valor ordinario de la distribución.

## 8. ¿Qué variables se relacionan más con `MEDV`?

### Pregunta
**¿Qué predictores muestran una asociación lineal más fuerte con la variable objetivo?**

En lugar de imprimir una matriz completa, calcularemos la correlación de cada predictor con `MEDV` y la ordenaremos visualmente.

> La correlación mide **asociación lineal**. No demuestra causalidad y tampoco garantiza que la relación sea perfectamente recta.

In [ ]:
corr_medv = (
    df.corr(numeric_only=True)["MEDV"]
      .drop("MEDV")
      .sort_values()
      .rename("Correlación")
      .reset_index().rename(columns={"index": "Variable"})
)

plt.figure(figsize=(8, 6))
sns.barplot(data=corr_medv, x="Correlación", y="Variable")
plt.axvline(0, linewidth=1)
plt.xlim(-1, 1)
plt.title("Correlación de cada predictor con MEDV")
plt.xlabel("Correlación de Pearson")
plt.ylabel("")
plt.show()

### Lectura del gráfico

- **`LSTAT`** presenta la asociación lineal negativa de mayor magnitud con `MEDV` (aprox. **−0.74**).
- **`RM`** presenta la asociación positiva más intensa (aprox. **+0.70**).
- `PTRATIO`, `INDUS`, `TAX` y `NOX` también muestran asociaciones negativas relevantes.

Pero una barra de correlación todavía no nos dice **cómo** es la relación. Para eso necesitamos mirar los puntos.

### 8.1 Forma de las relaciones más importantes

La siguiente visualización coloca `MEDV` frente a varios predictores relevantes. La nube de puntos permite detectar tendencias, dispersión y posibles relaciones no lineales.

In [ ]:
g = sns.pairplot(
    df,
    x_vars=["LSTAT", "RM", "PTRATIO", "NOX"],
    y_vars=["MEDV"],
    kind="reg",
    height=3.0,
    aspect=1.05,
    plot_kws={"scatter_kws": {"alpha": 0.55, "s": 28}, "line_kws": {"linewidth": 2}}
)
g.fig.suptitle("Predictores seleccionados frente a MEDV", y=1.05)
plt.show()

### Interpretación

- Cuando `LSTAT` aumenta, `MEDV` tiende a disminuir; además, la nube sugiere que la relación no es perfectamente lineal.
- Cuando `RM` aumenta, `MEDV` tiende a aumentar, aunque existe dispersión y un grupo de observaciones limitado en `MEDV = 50`.
- `PTRATIO` y `NOX` muestran tendencias negativas menos intensas.

Esto ilustra por qué **correlación + scatterplot** ofrecen una lectura más completa que una matriz numérica aislada.

### 8.2 Relación no lineal: `LSTAT` y `MEDV`

Una correlación resume la intensidad lineal con un solo número. Una curva suavizada puede revelar estructura que ese número no describe.

In [ ]:
plt.figure(figsize=(8.5, 5))
sns.scatterplot(data=df, x="LSTAT", y="MEDV", alpha=0.55)
sns.regplot(data=df, x="LSTAT", y="MEDV", scatter=False)
plt.title("LSTAT vs MEDV: tendencia suavizada")
plt.xlabel("LSTAT")
plt.ylabel("MEDV")
plt.show()

La curvatura visible es una advertencia útil: aunque `LSTAT` tenga una correlación fuerte, un modelo estrictamente lineal puede no capturar toda la estructura de esta relación.

### 8.3 Las variables discretas deben compararse como grupos

Para `CHAS` y `RAD` no es natural interpretar una nube continua. Es más claro comparar la distribución de `MEDV` dentro de cada categoría o nivel.

In [ ]:
plt.figure(figsize=(6.5, 4.5))
sns.violinplot(data=df, x="CHAS", y="MEDV", inner="quartile")
plt.title("Distribución de MEDV según CHAS")
plt.xlabel("CHAS")
plt.ylabel("MEDV")
plt.show()

plt.figure(figsize=(9, 4.5))
sns.boxplot(data=df, x="RAD", y="MEDV")
plt.title("Distribución de MEDV según el nivel de RAD")
plt.xlabel("RAD")
plt.ylabel("MEDV")
plt.show()

Estos gráficos permiten comparar grupos sin forzar la idea de una relación continua. También dejan ver que el número de observaciones y la dispersión varían entre categorías.

## 9. Relaciones entre predictores

### Pregunta
**¿Existen variables que se mueven de manera muy parecida y podrían estar aportando información redundante?**

Ahora sí es útil una matriz, pero presentada como **heatmap**. El color permite detectar rápidamente bloques de asociación sin leer decenas de números.

In [ ]:
corr_pred = df.drop(columns="MEDV").corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_pred,
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.4
)
plt.title("Mapa de correlaciones entre predictores")
plt.show()

### Patrones que conviene observar

- `RAD` y `TAX` presentan una asociación positiva muy alta (aprox. **0.91**).
- `NOX` y `DIS` muestran una asociación negativa fuerte (aprox. **−0.77**).
- `INDUS` y `NOX` están fuertemente asociadas de forma positiva.
- `AGE` y `DIS` presentan una asociación negativa importante.

Estas relaciones sugieren **redundancia potencial**, pero el EDA por sí solo no justifica eliminar variables. Esa decisión dependerá del modelo, del objetivo y de la validación posterior.

### 9.1 Un ejemplo visible de redundancia: `RAD` y `TAX`

In [ ]:
plt.figure(figsize=(9, 4.5))
sns.boxplot(data=df, x="RAD", y="TAX")
plt.title("TAX cambia fuertemente entre los niveles de RAD")
plt.xlabel("RAD")
plt.ylabel("TAX")
plt.show()

El patrón entre `RAD` y `TAX` ayuda a entender por qué dos variables pueden terminar aportando información muy parecida a un modelo.

## 10. Síntesis del EDA: ¿qué aprendimos antes de modelar?

El análisis exploratorio permite construir una imagen coherente del conjunto antes de entrenar cualquier algoritmo:

1. **Calidad básica:** el conjunto estándar contiene **506 observaciones y 14 variables**, sin valores faltantes ni filas duplicadas.
2. **Tipos de variables:** no todas las columnas numéricas deben tratarse como continuas. `CHAS` es binaria y `RAD` es discreta.
3. **Distribuciones:** varias variables son asimétricas. `CRIM` es el caso más evidente y `ZN` concentra muchas observaciones en cero.
4. **Valores extremos:** los boxplots detectan observaciones alejadas del rango central, pero un punto extremo **no debe eliminarse automáticamente**.
5. **Variable objetivo:** `MEDV` presenta **16 observaciones exactamente en 50**, una señal de efecto de techo que debe recordarse al evaluar modelos.
6. **Predictores asociados a `MEDV`:** `LSTAT` muestra la asociación lineal negativa más intensa y `RM` la positiva más intensa.
7. **Forma de la relación:** algunos scatterplots, especialmente `LSTAT`–`MEDV`, sugieren que la relación no es completamente lineal.
8. **Redundancia entre predictores:** `RAD`–`TAX`, `NOX`–`DIS`, `INDUS`–`NOX` y `AGE`–`DIS` muestran asociaciones fuertes.

### ¿Qué implica esto para el siguiente notebook?

Antes de construir un modelo será razonable estudiar:

- separación entre predictores y variable objetivo;
- partición entrenamiento/prueba;
- necesidad de escalado según el algoritmo;
- posibles transformaciones para variables muy asimétricas;
- efecto de predictores altamente correlacionados;
- comparación entre un modelo lineal de referencia y modelos capaces de representar relaciones no lineales.

> **Conclusión:** el EDA no nos dice todavía cuál es el mejor modelo, pero sí nos indica qué características del conjunto podrían afectar su comportamiento y qué decisiones tendremos que justificar posteriormente.